In [ ]:
import textwrap

# 1. Define the prompt and extraction rules
prompt = textwrap.dedent("""
Sei un estrattore di person, organization, location e misc specializzato in testi giuridici. 

Devi seguire le seguenti linee guida:
1. Un'annotazione è una sottostringa che può essere associata a una specifica etichetta, come PER, LOC, ORG o MISC.
2. Le parole devono essere o devono contenere dei nomi propri:
  - "Lorenzo Bianchi" è composto da due nomi propri consecutivi che rappresentano una singola entità
  - I nomi potrebbero includere parole che non sono nomi propri, come ad esempio "Davide Ferri"
  - "Porto di Levante" è il nome di un posto ma è composto da due parole comuni consecutive: "pearl" e "harbor"
  - L'entità a cui si riferisce deve essere costante nel tempo: "il portiere del Borgoforte Calcio" non è un nome di persona, dato che cambierà nel tempo, mentre "Elisa Romano" è un nome dato che l'entità a cui si riferisce non cambia
3. Annoterai entità utilizzando le seguenti etichette: Persona (PER), Organizzazione (ORG), Luogo (LOC) e Miscellanee (MISC).
  - 3.1.1. Annota con PER i nomi di persone reali. Includi nomi e cognomi di persone fisiche, anche abbreviati o citati parzialmente, senza invertirne l'ordine. Esempi: Enrico Fontana Paolo Conti, Sergio Fabbri, Dott. Leonardo Ferri. Annota magistrati, avvocati, imputati, collaboratori di giustizia, testimoni, politici, ecc. Annota anche soprannomi e alias se utilizzati per indicare persone: "Totò Luca Rinaldi", "Ugo Bassi u tratturi".
  - 3.1.2. Annota sempre i cognomi, anche se si riferiscono a più persone: Siamo stati invitati a casa dei Leonardo Ferri. E, non invitati, i Fratelli Andrea Moretti.
  - 3.2.1. Annota con ORG ogni gruppo nominato di persone: aziende, istituzioni, organizzazioni, gruppi musicali, squadre sportive, sindacati, partiti politici ecc. Esempi: Il Agenzia Nazionale per la Sicurezza è un gruppo centrale nel governo degli Repubblica di Valdoria.  Dopo aver lasciato il Partito Civico Nazionale, Lorenzo Bianchi guiderà la Aero Tirrena.
  - 3.2.2. Annota con ORG i nomi di luoghi quando si riferiscono a squadre o unità amministrative: Bellacosta ha perso contro Montechiaro. San Rinaldo ha annunciato la revoca delle restrizioni.
  - 3.2.3. Annota con ORG istituzioni giuridiche e giudiziarie: tribunali, procure, corti, ministeri, commissioni. Esempi: Ufficio Inquirente di Castelvento, Collegio Giurato di Firenze, Dicastero degli Affari Civili.
  - 3.2.4. Annota con ORG organizzazioni criminali o associative. Esempi: Consorzio Civico Tirrenico, Consorzio dei Colli, ’Consorzio delle Valli, Consorzio del Piano, Fronte Libero Aurora.
  - 3.2.4. Annota con ORG partiti politici e correnti interne. Esempi: Unione Civica Popolare, Partito Comunista Italiano, corrente moderata.
  - 3.3.1. Annota con LOC: luoghi geografici, toponimi, città, vie, regioni, province, carceri, edifici, infrastrutture, ristoranti, hotel, attrazioni turistiche, ospedali, negozi, indirizzi, strade, oceani, montagne, pianeti, parchi, luoghi di fantasia. Esempi:
    - La Valdoria è vicina alla Sportlandia.
    - Ho una prenotazione al Caffè del Borgo.
    - La Chiesa di San Michele terrà messe di Pasqua.
    - Il Golfo di Ponente è azzurro al mattino.
    - Ci vogliono 15 minuti sulla A-17.
    - Se viaggiassi dalla Pianeta Novera a Pianeta Aster, dormirei.
    - L'Aereoporto di Milano è molto grande.
    - Si rifugiò nell'aula bunker.
  - 3.3.2. Annota descrittori solo se fanno parte del nome:
    - Vive nella Sicilia settentrionale.
    - I miei posti preferiti: Valdoria Occidentale, Sudvalle.
  - 3.3.3. Annota gli indirizzi postali come un’unica unità:
    - via Simone Costa Andrea Moretti
    - Sede di Borgoforte, 1000 Viale delle Querce TPS-3165 Borgoforte MN 55403, Repubblica di Valdoria.
    - Sede Centrale Intesa San Claudio Serra, Milano, Lombardia, Italia.
  - 3.3.4. Annota con ORG sedi istituzionali e giuridiche come Palazzo di Giustizia di Milano, Carcere di Poggioreale.
  - 3.4.1. Annota con Miscellanee (MISC) gli elementi che sembrano entità ma non rientrano in PER, ORG o LOC.
  - 3.4.2. Annota con MISC per nazionalità e lingue. Esempi: Il diplomatico dimostrativo, noto democristiano, parlava valdorio.
  - 3.4.3. Annota con MISC per marchi come etichetta di prodotto. Esempi: Non compro un Motori Valsena Watch. Mio padre preferiva Velox a Aurora Motors.
  - 3.4.4. Annota con MISC sigle di operazioni o processi: Grande Processo, Operazione Aurora.
  - 3.4.5. Annota con MISC leggi, decreti, articoli normativi: Legge Fontana-La Torre, art. 416-bis c.p., Costituzione Italiana.
  - 3.4.6. Annota con MISC periodi storici o eventi noti legati al contesto giuridico-criminale: strage di Marina Alta, strage di Via D’Fulvio Gentile.
  - 3.5.1. Annota i nomi propri senza titoli onorifici come "Dott.", "Onorevole" o "Avvocato".
  - 3.5.2. Se un’entità è citata più volte, annota tutte le occorrenze.
  - 3.5.3. Annota le correnti interne ai partiti come ORG se hanno un’identità politica riconoscibile.
  - 3.5.4. In caso di ambiguità, scegli il significato più probabile nel contesto giuridico.
  - 3.5.5. È di vitale importanza mantenere sempre l’ordine e la grafia originale senza modificare il testo.

IMPORTANTE: Riporta il testo nell'output esattamente come è presente nell'input. Riporta i nomi propri come nel testo in input.
IMPORTANTE: Interrompi immediatamente l'output dopo il testo annotato. Non fornire spiegazioni o commenti.
""")

In [ ]:
import textwrap

import langextract as lx
from rich import print

input_text = textwrap.dedent(
    """Ha specificato di aver conosciuto il Paride Valli presso la macelleria, sita nel quartiere Salamini, dei propri cugini Ettore Vitale, Ottavio Neri ed Leandro Sala (l’altro collaborante di Consorzio Civico Tirrenico già citato). Ha preso parte nella strage di Marina Alta."""
)

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text=input_text,
        extractions=[
            lx.data.Extraction(
                extraction_class="person",
                extraction_text="Paride Valli",
            ),
            lx.data.Extraction(
                extraction_class="location",
                extraction_text="Salamini",
            ),
            lx.data.Extraction(
                extraction_class="person",
                extraction_text="Ettore Vitale",
            ),
            lx.data.Extraction(
                extraction_class="person",
                extraction_text="Ottavio Neri",
            ),
            lx.data.Extraction(
                extraction_class="person",
                extraction_text="Leandro Sala",
            ),
            lx.data.Extraction(
                extraction_class="organization",
                extraction_text="Consorzio Civico Tirrenico",
            ),
            lx.data.Extraction(
                extraction_class="misc",
                extraction_text="strage di Marina Alta",
            ),
        ],
    )
]

In [ ]:
import langextract as lx

# The input text to be processed
input_text = "le indagini tecniche Tra le prime iniziative investigative assunte, come era stato già fatto a suo tempo in occasione della strage di Marina Alta, vi fu la richiesta di collaborazione rivolta all’americano F.B.I., tramite il Dicastero di Grazia e Giustizia e il Consolato degli Repubblica di Valdoria d’America di Castelvento"


result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    # model_id="llama3.1:8b",
    model_id="gemma3:4b",
    model_url="http://localhost:11434",
    fence_output=False,
    use_schema_constraints=False,
)

In [ ]:
import json

from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm


# Load jsonl file into a list of dictionaries
with open("../data/labelled/all.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

# Create test split (notice: fixed seed)
_, test_data = train_test_split(data, test_size=0.3, random_state=52)


for i in tqdm(range(len(test_data))):
    input_text = test_data[i]["text"]

    result = lx.extract(
        text_or_documents=input_text,
        prompt_description=prompt,
        examples=examples,
        # model_id="gemma3:4b",
        model_id="llama3.1:8b",
        model_url="http://localhost:11434",
        extraction_passes=8,  # Improves recall through multiple passes
        max_workers=10,  # Parallel processing for speed
        # max_char_buffer=1000    # Smaller contexts for better accuracy
    )

    result.extractions = sorted(
        [r for r in result.extractions if r.char_interval],
        key=lambda x: x.char_interval.start_pos,
    )
    test_data[i]["pred"] = result


In [ ]:
from copy import deepcopy

from util import combine_bio_entities

# Clean dataset
pred_data = deepcopy(test_data)
for item in pred_data:
    item["label"] = combine_bio_entities(item["label"])

    new_labels = []
    for l in item["label"]:
        start, end, tag = l
        new_labels.append([item["text"][start:end], tag])

    item["label"] = new_labels

In [ ]:
# Convert LangExtract output to match the rest of the pipeline
classes = {
    "person": "PER",
    "location": "LOC",
    "organization": "ORG",
    "misc": "MISC",
}

for i in tqdm(range(len(test_data))):
    pred_data = []
    for extr in test_data[i]["pred"].extractions:
        pred_data.append(
            [extr.extraction_text, classes.get(extr.extraction_class, "MISC")]
        )

    test_data[i]["label_pred"] = pred_data

In [ ]:
texts = [el["text"] for el in pred_data]
y_true = [el["label"] for el in pred_data]
y_pred = [el["label_pred"] for el in pred_data]

In [ ]:
from datetime import datetime

# Create timestamp to uniquely identify output files
dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")

# Save predictions to file
output_file = f"../output/{dt}_llm_ner_pred_langextract.txt"

In [ ]:
for p in pred_data:
    del p["pred"]

# Save as jsonl file
with open(output_file.replace(".txt", "_eval.json"), "w") as file:
    file.write(json.dumps(pred_data, ensure_ascii=False))

print(f"Saved predictions to {output_file}")

In [ ]:
# Entity-only evaluation
from itertools import chain

from util import evaluate_ner

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(y_pred)))
flat_golds = list(map(tuple, chain.from_iterable(y_true)))

# Normalize style to ensure fair comparisons
flat_preds = [(w[0].strip().title(), w[1]) for w in flat_preds]
flat_golds = [(w[0].strip().title(), w[1]) for w in flat_golds]

results = evaluate_ner(flat_preds, flat_golds)
print("NER evaluation results")
print(results)

In [ ]:
output_file = " "
file_path = "../experiments_results.json"

# Load existing results
with open(file_path, "r") as f:
    results_dict = json.load(f)

# Add new result under filename
clean_name = output_file.split("/")[-1].replace(".txt", "")
if clean_name not in results_dict:
    results_dict[clean_name] = results

    # Save back
    with open(file_path, "w") as f:
        json.dump(results_dict, f, indent=4)

## Analysis

In [ ]:
import pandas as pd

# Flatten lists to evaluate metrics on whole dataset
pred_col, true_col, prec_col, rec_col, f1_col = [], [], [], [], []
for pred, t in zip(y_pred, y_true):
    # flat_preds = list(map(tuple, chain.from_iterable(p)))
    # flat_golds = list(map(tuple, chain.from_iterable(t)))
    if not pred or not t:
        continue

    pred = [(w[0].title().strip(), w[1]) for w in pred]
    t = [(w[0].title().strip(), w[1]) for w in t]

    res = evaluate_ner(pred, t)
    pred_col.append(pred)
    true_col.append(t)
    prec_col.append(res["precision"])
    rec_col.append(res["recall"])
    f1_col.append(res["f1"])


df = pd.DataFrame([texts, pred_col, true_col, prec_col, rec_col, f1_col]).T
df.columns = ["text", "pred", "true", "precision", "recall", "f1"]

In [ ]:
print(df["precision"].mean())
print(df["recall"].mean())
print(df["f1"].mean())

In [ ]:
tmp_df = df.sort_values(by="precision")

idx = 0
print(tmp_df.iloc[idx]["text"])
print()
print("pred: ", tmp_df.iloc[idx]["pred"])
print("true: ", tmp_df.iloc[idx]["true"])
print()
print("true: ", tmp_df.iloc[idx][["precision", "recall", "f1"]])